<a href="https://colab.research.google.com/github/Aiman-Naheed-Iqbal/ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aiman-Naheed-Iqbal/ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [14]:
import pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)
df = pd.read_parquet(path)
print(df.shape)

(9841378, 30)


In [15]:
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)
dim_content = pd.read_parquet(dim_content_path)
print(dim_content.shape)

(519606, 26)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [16]:
# Filter to rows where GSC data actually exists
march = df[df['gsc_data_available'] == True].copy()

# Merge in static content-level fields
features = march.merge(
    dim_content[['client_hash_id', 'content_hash_id', 'search_volume',
                 'competition', 'cpc', 'backlinks', 'keyword_char_count',
                 'content_type', 'main_intent']],
    on=['client_hash_id', 'content_hash_id'],
    how='left'
)

# Five features
feature_frame = pd.DataFrame({
    'client_hash_id': features['client_hash_id'],
    'content_hash_id': features['content_hash_id'],
    'report_date': features['report_date'],
    'gsc_impressions': features['gsc_impressions'],
    'search_volume': features['search_volume'],
    'competition': features['competition'],
    'cpc': features['cpc'],
    'backlinks': features['backlinks'],
})

# Fill missing values
feature_frame = feature_frame.fillna({
    'search_volume': 0, 'competition': 0, 'cpc': 0, 'backlinks': 0
})

print(feature_frame.shape)
feature_frame.head()

(3611061, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,search_volume,competition,cpc,backlinks
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,20.0,0.00,0.00,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,10.0,1.00,0.00,0.0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,20.0,0.00,0.00,0.0
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,90.0,1.00,1.01,0.0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,40.0,0.45,0.09,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [17]:
1. #gsc_impressions** — meaning: how many times the page appeared in search results that day. Available at decision time because it's logged same-day by Search Console; no future data used. No missing values (filtered to gsc_data_available == True).

2. #search_volume** — meaning: estimated monthly searches for the keyword. Knowable in advance — a keyword-level estimate independent of any specific day's outcome. Missing values filled with 0 (keyword not yet scored by provider).

3. #competition** — meaning: how contested the keyword is in paid/organic search. Available before the fact — static keyword attribute, not derived from ranking outcomes. Missing filled with 0.

4. #cpc** — meaning: cost-per-click, a market pricing signal for the keyword. Exists prior to and independent of ranking outcomes. Missing filled with 0.

5. #backlinks** — meaning: number of backlinks to the content/page. Reflects link count as of crawl time, not derived from future ranking movement. Missing filled with 0.

#Categorical fields (main_intent, content_type) would need one-hot encoding before modeling — noted for later, not required here.


5.0

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [18]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Sample down for speed — leakage demo doesn't need millions of rows
sample = feature_frame.sample(n=50000, random_state=42).copy()

sample['label'] = (sample['gsc_impressions'] > sample['gsc_impressions'].median()).astype(int)

honest_features = ['search_volume', 'competition', 'cpc', 'backlinks']
X = sample[honest_features].fillna(0)
y = sample['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
honest_score = accuracy_score(y_test, model.predict(X_test))
print("HONEST score (no leakage):", honest_score)

# --- Deliberate leak ---
sample['leaky_gsc_impressions'] = sample['gsc_impressions']
leaky_features = honest_features + ['leaky_gsc_impressions']
X_leak = sample[leaky_features].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.2, random_state=42)
model_leak = RandomForestClassifier(n_estimators=50, random_state=42)
model_leak.fit(X_train, y_train)
leaky_score = accuracy_score(y_test, model_leak.predict(X_test))
print("LEAKY score (with label-derived column):", leaky_score)

print(f"\nScore jumped from {honest_score:.3f} to {leaky_score:.3f} once the leaky column was added.")
print("Deleting leaky_gsc_impressions — it directly encodes the label, so it's not a real feature.")

HONEST score (no leakage): 0.5588
LEAKY score (with label-derived column): 1.0

Score jumped from 0.559 to 1.000 once the leaky column was added.
Deleting leaky_gsc_impressions — it directly encodes the label, so it's not a real feature.


In [19]:
#Excluded fields and why:**

#client_hash_id / content_hash_id / keyword_hash_id / url_hash_id** — identifier columns, not predictive features; used only for joining tables, not fed into the model.
#leaky_gsc_impressions** (the deliberate leak) — directly derived from the label itself; including it lets the model "cheat" by seeing the answer, so it's removed permanently.
#Branded queries** — excluded from the feature set because their near-guaranteed top ranking distorts the opportunity signal for the actual keywords being scored.
#ga4_data_available fields** — excluded for this slice since coverage is much lower than GSC (many clients have `client_has_ga4 == False`), so including GA4-derived features would introduce large amounts of missing data specific to a subset of clients.
#content_created_date / content_updated_date** — excluded as direct features since they're better suited as filters (e.g. content age) than raw model inputs at this stage; noted for future feature engineering.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.